# pctrl: staged minimization of two-photon basis counts

This example uses `qtools.pctrl.optimize.BasisMeasurement` and `pctrl.optimize()`.
Each `optimize()` call performs exactly one stage. Coarse, fine, and final
passes are separate calls, with each result used as the next warm start.

Motor connection settings come from `motor_config.yaml`; basis calibration
angles come from `polmeas_config.yaml`.


## 1. Configuration

Launch Jupyter from the experiment directory and set the TDC window
parameters for the current setup. `TARGET` can be one channel (one-sided
optimization) or `["alice", "bob"]` (joint optimization).


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt

from qtools.motor import MotorConfigManager
from qtools.pctrl import BasisMeasurement, PolarizationController
from qtools.pmeas.config import ConfigManager as PolmeasConfigManager
from qtools.pmeas.motor_manager import MotorManager

PCTRL_CONFIG = Path.cwd() / "pctrl_config.yaml"
MOTOR_CONFIG = Path.cwd() / "motor_config.yaml"
POLMEAS_CONFIG = Path.cwd() / "polmeas_config.yaml"
TARGET = ["alice", "bob"]  # use "alice" for one-sided optimization

MINIMIZATION_BASIS = [("H", "V"), ("V", "H"), ("D", "A"), ("A", "D")]
MAXIMIZATION_BASIS = []

CH_START, CH_STOP = 1, 4
DELAY = 100
WINDOW_START, WINDOW_STOP = 109, 110
THRESHVOLT = 0.6

SPEED_DEG_S = 10.0
SETTLE_TIME = 0.2
OVERSHOOT = True

# Each entry is one independent optimize() call.
# The previous stage result becomes the next stage warm start.
STAGES = [
    {
        "name": "coarse",
        "search_radius_deg": None,  # full mechanical range
        "step_size_deg": 60.0,
        "maxiter": 40,
        "tol": 2.0,
        "duration": 0.2,
        "samples": 1,
    },
    {
        "name": "fine",
        "search_radius_deg": 20.0,
        "step_size_deg": 10.0,
        "maxiter": 40,
        "tol": 0.8,
        "duration": 0.5,
        "samples": 2,
    },
    {
        "name": "final",
        "search_radius_deg": 6.0,
        "step_size_deg": 3.0,
        "maxiter": 25,
        "tol": 0.3,  # one SCS encoder step is about 0.2933 deg
        "duration": 1.0,
        "samples": 3,
    },
]

for path in (PCTRL_CONFIG, MOTOR_CONFIG, POLMEAS_CONFIG):
    print(f"{path}: {'OK' if path.exists() else 'missing'}")


## 2. Create the controller and basis measurement

`BasisMeasurement` switches the Alice/Bob basis and reads the TDC;
`pctrl.optimize()` owns the waveplate optimization loop. The loss is built
automatically from the basis lists:

`loss = sum(rate**2 for rate in MINIMIZATION_BASIS) - sum(rate**2 for rate in MAXIMIZATION_BASIS)`


In [ ]:
pctrl = PolarizationController(config_path=str(PCTRL_CONFIG))
motor_cfg = MotorConfigManager.resolve(MOTOR_CONFIG)
polmeas_cfg = PolmeasConfigManager.resolve(POLMEAS_CONFIG)
motor_mgr = MotorManager(motor_cfg)

basis_measurement = BasisMeasurement(
    meas_mgr=pctrl.meas_mgr,
    polmeas_cfg=polmeas_cfg,
    motor_mgr=motor_mgr,
    minimization_basis=MINIMIZATION_BASIS,
    maximization_basis=MAXIMIZATION_BASIS,
)
print("controllers ready")


## 3. Run staged pctrl.optimize calls

Do not hide the three stages inside one call. Here `optimize()` is invoked
once per stage:

1. Coarse search uses the full mechanical range and a large simplex step.
2. Fine search uses the coarse result as `initial_angles` and limits the range
   with `search_radius_deg`.
3. Final polish uses a smaller radius and step, plus longer integration and
   more samples.

Across separate processes, use `initial_angles="saved"` to load the
`optimal_phys_angle` values previously stored with `save_config=True`.


In [ ]:
def progress(iteration, angles, value, _raw):
    basis_measurement.history_rows.append((
        len(basis_measurement.history_rows) + 1,
        dict(basis_measurement.last_counts),
        float(value),
    ))
    counts = " ".join(
        f"{a}{b.lower()}={basis_measurement.last_counts[(a, b)]:.2f}"
        for a, b in basis_measurement.measurement_keys
    )
    print(f"eval {len(basis_measurement.history_rows):03d} | loss={value:.3f} | {counts}")

stage_results = {}
stage_boundaries = []
previous_angles = None

for stage in STAGES:
    print(f"\n=== {stage['name']} stage ===")
    stage_result = pctrl.optimize(
        target=TARGET,
        measurement_adapter=basis_measurement,
        minimize=True,
        method="Nelder-Mead",
        initial_angles=previous_angles,
        search_radius_deg=stage["search_radius_deg"],
        step_size_deg=stage["step_size_deg"],
        maxiter=stage["maxiter"],
        tol=stage["tol"],
        speed_deg_s=SPEED_DEG_S,
        settle_time=SETTLE_TIME,
        overshoot=OVERSHOOT,
        duration=stage["duration"],
        samples=stage["samples"],
        measure_type="pairs",
        ch_start=CH_START,
        ch_stop=CH_STOP,
        delay=DELAY,
        window_start=WINDOW_START,
        window_stop=WINDOW_STOP,
        threshvolt=THRESHVOLT,
        callback=progress,
        verbose=True,
    )
    stage_results[stage["name"]] = stage_result
    stage_boundaries.append(len(basis_measurement.history_rows))
    previous_angles = stage_result.optimal_angles
    print(stage_result.message)
    print("optimal angles:", stage_result.optimal_angles)

final = stage_results["final"]
print("\nfinal warm start:", final.initial_angles)
print("final search bounds:", final.search_bounds)
print("final optimal angles:", final.optimal_angles)


## 4. Inspect basis counts and convergence

`BasisMeasurement` keeps the evaluation history. Vertical lines mark the
coarse, fine, and final stage boundaries.


In [ ]:
rows = basis_measurement.history_rows
labels = ["".join(key) for key in basis_measurement.measurement_keys]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot([row[0] for row in rows], [row[2] for row in rows], marker="o")
axes[0].set(xlabel="evaluation", ylabel="loss", title="basis loss")
for boundary in stage_boundaries[:-1]:
    axes[0].axvline(boundary + 0.5, color="gray", linestyle="--", alpha=0.5)
axes[0].grid(True, alpha=0.3)

for index, label in enumerate(labels):
    key = basis_measurement.measurement_keys[index]
    axes[1].plot(
        [row[0] for row in rows],
        [row[1][key] for row in rows],
        marker="o",
        label=label,
    )
for boundary in stage_boundaries[:-1]:
    axes[1].axvline(boundary + 0.5, color="gray", linestyle="--", alpha=0.5)
axes[1].set(xlabel="evaluation", ylabel="coincidence rate (cps)", title="basis rates")
axes[1].legend()
axes[1].grid(True, alpha=0.3)
plt.tight_layout()


## 5. Close connections

Run this after the experiment.


In [ ]:
try:
    pctrl.close()
finally:
    motor_mgr.disconnect_all()
print("hardware connections closed")
